# Question 1

###  Install uv
(base) root@MS-CEVXSRKPSHOI:/mnt/d/develop/mlops/ml# curl -LsSf https://astral.sh/uv/install.sh | sh
downloading uv 0.12.5 x86_64-unknown-linux-gnu
installing to /root/.local/bin
  uv
  uvx
everything's installed!

To add $HOME/.local/bin to your PATH, either restart your shell or run:

    source $HOME/.local/bin/env (sh, bash, zsh)
    source $HOME/.local/bin/env.fish (fish)
### 添加环境变量
source $HOME/.local/bin/env

### What's the version of uv you installed?

In [3]:
# Use --version to find out
(base) root@MS-CEVXSRKPSHOI:~/.local/bin# uv --version
uv 0.12.5 (x86_64-unknown-linux-gnu)

In [ ]:
# Initialize an empty uv project
# You should create an empty folder for homework and do it there. 
(base) root@MS-CEVXSRKPSHOI:/mnt/d/develop/mlops/ml# uv init  ml_project_uv
Initialized project `ml-project-uv` at `/mnt/d/develop/mlops/ml/ml_project_uv`
# 创建虚拟环境
uv venv

Using CPython 3.13.13 interpreter at: /root/miniconda3/bin/python3
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
# 激活虚拟环境
source .venv/bin/activate
# 退出虚拟环境
deactivate

# Question 2

In [ ]:
# Use uv to install Scikit-Learn version 1.6.1
# 现在可以安装包了
uv add scikit-learn==1.6.1
uv add pandas

In [ ]:
# What's the first hash for Scikit-Learn you get in the lock file?
# Include the entire string starting with sha256:, don't include quotes
sha256:8561a3269e6801106863fd0d6d84bb737be9e7631e33aaed3fb9ce5953688da3

# Question 3

In [ ]:
0.533

# Question 4

In [ ]:
# 安装 FastAPI 和 Uvicorn（ASGI 服务器）
uv add fastapi uvicorn  
uv add requests

In [ ]:
# serve.py

import pickle
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import pandas as pd
from typing import Dict, Any

# 1. 加载模型
MODEL_PATH = "pipeline_v1.bin"  # 确保路径正确

try:
    with open(MODEL_PATH, "rb") as f:
        pipeline = pickle.load(f)
    print(f"✅ Model loaded successfully from {MODEL_PATH}")
except FileNotFoundError:
    print(f"❌ Model file not found: {MODEL_PATH}")
    print("Please ensure pipeline_v1.bin is in the current directory")
    exit(1)

# 2. 定义请求体数据结构（Pydantic 模型）
# ClientData 继承自 BaseModel，它的作用是：
# 定义数据结构：规定 API 请求体必须包含哪些字段，以及它们的类型
# 自动验证：如果传入的数据类型不对（比如 annual_income 传了字符串），它会自动报错
# 自动生成文档：FastAPI 会根据这个定义生成 Swagger 文档
class ClientData(BaseModel):
    lead_source: str
    number_of_courses_viewed: int
    annual_income: float
    
    # 可选：添加数据验证,为 API 文档提供一个示例数据,当你访问 /docs（Swagger UI）时，会自动填充这个示例
    class Config:
        json_schema_extra = {
            "example": {
                "lead_source": "organic_search",
                "number_of_courses_viewed": 4,
                "annual_income": 80304.0
            }
        }


# 3. 创建 FastAPI 应用
app = FastAPI(
    title="Lead Scoring Model API",
    description="Predict whether a lead will convert",
    version="1.0.0"
)

# 4. 健康检查端点
@app.get("/")
def read_root():
    return {"message": "Lead Scoring Model API is running", "status": "healthy"}

# 5. 预测端点
@app.post("/predict")
def predict(client: ClientData):
    """
    Predict conversion probability for a single client
    """
    try:
        # 将输入数据转换为字典
        input_dict = client.model_dump()  # Pydantic v2
        # 如果是 Pydantic v1，用 .dict()
        
        # 转换为字典列表（DictVectorizer 需要的格式）
        input_data = [input_dict]
        
        # 进行预测
        prediction = pipeline.predict(input_data)[0]
        
        # 获取概率（如果是分类模型）
        try:
            probability = pipeline.predict_proba(input_data)[0][1]  # 假设是二分类
        except (AttributeError, IndexError):
            probability = None
        
        # 返回结果
        response = {
            "prediction": int(prediction),
            "probability": float(probability) if probability is not None else None,
            "input_data": input_dict
        }
        return response
        
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction error: {str(e)}")


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=9009) 

In [ ]:
# 启动API服务
uv run uvicorn serve:app --host 0.0.0.0 --port 9009 --reload

INFO:     Will watch for changes in these directories: ['/mnt/d/develop/mlops/ml/ml_project_uv']
INFO:     Uvicorn running on http://0.0.0.0:9009 (Press CTRL+C to quit)
INFO:     Started reloader process [69651] using StatReload
✅ Model loaded successfully from pipeline_v1.bin
INFO:     Started server process [69672]
INFO:     Waiting for application startup.
INFO:     Application startup complete.

In [ ]:
# 测试web服务
# test_client.py

import requests
import json

# 服务地址
url = "http://127.0.0.1:9009/predict"

# 你的测试数据
client = {
    "lead_source": "organic_search",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0
}

# 发送 POST 请求
response = requests.post(url, json=client)

# 打印结果
print("Status Code:", response.status_code)
print("Response:", json.dumps(response.json(), indent=2))

In [ ]:
(ml_project_uv) (base) root@MS-CEVXSRKPSHOI:/mnt/d/develop/mlops/ml/ml_project_uv# python test_client.py 
Status Code: 200
Response: {
  "prediction": 1,
  "probability": 0.5340417283801275,
  "input_data": {
    "lead_source": "organic_search",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0
  }
}

What's the probability that this client will get a subscription?

0.534

# Question 5
Download the base image agrigorev/zoomcamp-model:2025. You can easily make it by using docker pull command.

So what's the size of this base image?

45 MB

In [ ]:
# 拉取镜像
docker pull agrigorev/zoomcamp-model:2025

In [ ]:
# Dockerfile
FROM agrigorev/zoomcamp-model:2025

WORKDIR /app

# 复制项目文件
COPY pyproject.toml README.md ./
COPY src/ ./src/

# 安装项目及其依赖，安装的依赖信息，直接来自于 pyproject.toml 文件
RUN pip install --no-cache-dir .

# 5. 复制应用代码
COPY serve.py ./
COPY pipeline_v1.bin ./

# 6. 暴露端口
EXPOSE 9008

# 7. 启动命令,
# 如果没有ENTRYPOINT——它就是容器启动时的唯一主命令。
# 如果有ENTRYPOINT——ENTRYPOINT就是容器启动时的唯一主命令，CMD则可以作为ENTRYPOINT的参数。
CMD ["uvicorn", "serve:app", "--host", "0.0.0.0", "--port", "9008"]

In [ ]:
# 创建名为：my-model-api 的镜像
docker build -t my-model-api:latest .

In [ ]:
docker run -it --rm -p 9008:9008 my-model-api:latest

# Question 6

In [ ]:
0.59